In [1]:
# Necessary Imports
import os
import cv2
import torch
from ultralytics import YOLO
from tqdm import tqdm
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models
from torchvision.transforms import v2
from torch.utils.data import DataLoader,Dataset
from torch.amp import autocast, GradScaler  # Using GradScaler because I was running into CUDA OOM error
from PIL import Image
import matplotlib.pyplot as plt
torch.manual_seed(37)

In [5]:
# YOLO Pipeline for generating cropped images of leaves
RAW_DATA_DIR = "Merged_Dataset/train"          # Path to your original folders
PROCESSED_DATA_DIR = "Merged_Dataset/cropped_training_380" # Where the new dataset will go
YOLO_MODEL = YOLO("yolo_v11_plant_doc.pt")        # Use your trained YOLO
IMG_SIZE = 380                        # Final size for EfficientNet

def prepare_training_data():
    if not os.path.exists(PROCESSED_DATA_DIR):
        os.makedirs(PROCESSED_DATA_DIR)

    classes = os.listdir(RAW_DATA_DIR)
    
    for cls in classes:
        input_class_path = os.path.join(RAW_DATA_DIR, cls)
        output_class_path = os.path.join(PROCESSED_DATA_DIR, cls)
        
        if not os.path.isdir(input_class_path): continue
        os.makedirs(output_class_path, exist_ok=True)
        
        print(f"Processing Class: {cls}")
        for img_name in tqdm(os.listdir(input_class_path)):
            img_path = os.path.join(input_class_path, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            
            # Run YOLO
            results = YOLO_MODEL(img, conf=0.3, verbose=False)
            boxes = results[0].boxes
            
            # Logic: Crop if found, else keep full
            if len(boxes) > 0:
                best_box = boxes[0].xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = map(int, best_box)
                processed_img = img[y1:y2, x1:x2]
            else:
                processed_img = img
            
            try:
                processed_img = cv2.resize(processed_img, (IMG_SIZE, IMG_SIZE))
                cv2.imwrite(os.path.join(output_class_path, img_name), processed_img)
            except:
                continue

prepare_training_data()

Processing Class: Potato_Early_blight


100%|██████████| 2274/2274 [00:47<00:00, 47.66it/s]


Processing Class: Potato_healthy


100%|██████████| 2073/2073 [00:42<00:00, 48.54it/s]


Processing Class: Potato_Lateblight


...

100%|██████████| 2023/2023 [00:42<00:00, 47.29it/s]


Processing Class: Tomato_Septoria_leaf_spot


100%|██████████| 2105/2105 [00:48<00:00, 43.29it/s]


Processing Class: Tomato_Tomato_Yellow_Leaf_Curl_Virus


100%|██████████| 2202/2202 [00:49<00:00, 44.69it/s]


In [13]:
# --- HYPERPARAMETERS ---
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 0.0001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_transforms = {
    'train': v2.Compose([
        v2.RandomEqualize(),
        v2.RandomHorizontalFlip(),
        v2.RandomRotation(degrees=(0,180)),
        v2.ToTensor(),
        v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': v2.Compose([
        v2.ToTensor(),
        v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# For applying different transformations to training and validation splits
class TransformedSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)

# Creating the dataset and loaders
full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training", transform=None)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# Applying appropriate transformations on the training as well as the validation dataset
train_data = TransformedSubset(train_set, transform=data_transforms['train'])
val_data = TransformedSubset(val_set, transform=data_transforms['val'])

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
full_loader = DataLoader(full_dataset,batch_size=BATCH_SIZE,shuffle=True)

# Setting up the model and modifying it's classification head
# model = models.efficientnet_b7(weights="DEFAULT")
# model = models.efficientnet_b4(weights="DEFAULT")
# model = models.efficientnet_b6(weights="DEFAULT")
model = models.convnext_base(weights="DEFAULT")
# model = models.convnext_small(weights="DEFAULT")
num_classes = len(full_dataset.classes)
# model.classifier[1] = nn.Linear(1792, num_classes)
# model.classifier[1] = nn.Linear(2560, num_classes)
# model.classifier[1] = nn.Linear(2304, num_classes)
model.classifier[2] = nn.Linear(1024, num_classes)
# model.classifier[2] = nn.Linear(1536, num_classes)
# model.classifier[2] = nn.Linear(768, num_classes)
model.to(DEVICE)

# Setting up the loss function, the optimizer, and scaler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = GradScaler() # Using GradScaler because I am using an RTX GPU for training, 
                      # might need to change this code accordingly.




c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [14]:
# Training Loop with tqdm
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]", leave=True)
    
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        # --- MIXED PRECISION FORWARD PASS ---
        with autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
        
        # --- BACKWARD PASS WITH SCALER ---
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} Completed. Average Loss: {avg_loss:.4f}")

Epoch [1/20]: 100%|██████████| 179/179 [00:39<00:00,  4.51it/s, loss=0.0582]


Epoch 1 Completed. Average Loss: 1.7335


Epoch [2/20]: 100%|██████████| 179/179 [00:28<00:00,  6.24it/s, loss=3.65] 


Epoch 2 Completed. Average Loss: 1.2581


Epoch [3/20]: 100%|██████████| 179/179 [00:24<00:00,  7.17it/s, loss=0.241]


...

Epoch 18 Completed. Average Loss: 0.2409


Epoch [19/20]: 100%|██████████| 179/179 [00:24<00:00,  7.35it/s, loss=0.331] 


Epoch 19 Completed. Average Loss: 0.2542


Epoch [20/20]: 100%|██████████| 179/179 [00:24<00:00,  7.23it/s, loss=0.849] 

Epoch 20 Completed. Average Loss: 0.2291


In [15]:
# Validation Block

def validate_model(model, loader, device):
    model.eval()
    correct_1 = 0
    correct_5 = 0
    total = 0
    all_preds = []
    all_labels = []

    print("Running Validation...")
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            
            # Top-1 Accuracy
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct_1 += (pred == labels).sum().item()

            # Top-5 Accuracy
            _, top5_preds = outputs.topk(5, 1, True, True)
            correct_5 += top5_preds.eq(labels.view(-1, 1).expand_as(top5_preds)).sum().item()

            # Store for Report
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    top1_acc = 100 * correct_1 / total
    top5_acc = 100 * correct_5 / total

    print(f"\n--- Validation Results ---")
    print(f"Top-1 Accuracy: {top1_acc:.2f}%")
    print(f"Top-5 Accuracy: {top5_acc:.2f}%")
    
    return all_labels, all_preds

# Run the validation
y_true, y_pred = validate_model(model, val_loader, DEVICE)

Running Validation...

--- Validation Results ---
Top-1 Accuracy: 58.20%
Top-5 Accuracy: 94.39%


In [20]:
# Testing Block

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

def final_test(model, test_path):
    model.eval()
    correct = 0
    total = 0
    correct_labels = []
    predicted_labels = []
    
    results_summary = {}

    print("Running validation on subset...")
    
    for class_folder in os.listdir(test_path):
        folder_path = os.path.join(test_path, class_folder)
        if not os.path.isdir(folder_path): continue
        
        results_summary[class_folder] = {"correct": 0, "total": 0}
        
        for img_name in tqdm(os.listdir(folder_path), desc=f"Testing {class_folder}"):

            img_path = os.path.join(folder_path, img_name)
            correct_labels.append(class_folder.strip())
            
            try:
                img = Image.open(img_path).convert('RGB')
            except:
                continue
                
            input_tensor = preprocess(img).unsqueeze(0).to(DEVICE)
            
            with torch.no_grad():
                output = model(input_tensor)
                _, pred_idx = torch.max(output, 1)
                
                predicted_name = labels[pred_idx.item()]
                predicted_labels.append(predicted_name.strip())
            
            if predicted_name.strip() == class_folder.strip():
                correct += 1
                results_summary[class_folder]["correct"] += 1
            
            total += 1
            results_summary[class_folder]["total"] += 1
    print(results_summary)

    print("\n" + "="*40)
    print(f"{'Class Name':<30} | {'Accuracy':<10}")
    print("-"*45)
    for cls, stats in results_summary.items():
        acc = (stats['correct']/stats['total'])*100 if stats['total'] > 0 else 0
        print(f"{cls:<30} | {acc:>8.2f}%")
    
    final_score = (correct / total) * 100
    print("="*40)
    print(f"OVERALL ACCURACY ON SUBSET: {final_score:.2f}%")
    return correct_labels, predicted_labels

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [18]:
# Best Model as of now
model_loaded = torch.load("ConVexNet-Base-Merged-Plant_Doc_Wild.pth", weights_only=False)
y_true, y_pred = final_test(model_loaded, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:01<00:00, 56.74it/s]

{'Potato_Early_blight': {'correct': 47, 'total': 87}, 'Potato_healthy': {'correct': 50, 'total': 63}, 'Potato_Lateblight': {'correct': 47, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 65, 'total': 100}, 'Tomato_Early_blight': {'correct': 31, 'total': 62}, 'Tomato_healthy': {'correct': 49, 'total': 71}, 'Tomato_Late_blight': {'correct': 54, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 56, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 50, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 58, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    54.02%
Potato_healthy                 |    79.37%
Potato_Lateblight              |    45.19%
Tomato_Bacterial_spot          |    65.00%
Tomato_Early_blight            |    50.00%
Tomato_healthy                 |    69.01%
Tomato_Late_blight             |    60.00%
Tomato_Leaf_mold               |    76.71%
Tomato_Septoria_leaf_s

In [19]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred,zero_division=0))

                                      precision    recall  f1-score   support

                 Potato_Early_blight       0.51      0.54      0.52        87
                   Potato_Lateblight       0.59      0.45      0.51       104
                      Potato_healthy       0.78      0.79      0.79        63
               Tomato_Bacterial_spot       0.58      0.65      0.61       100
                 Tomato_Early_blight       0.42      0.50      0.46        62
                  Tomato_Late_blight       0.60      0.60      0.60        90
                    Tomato_Leaf_mold       0.84      0.77      0.80        73
           Tomato_Septoria_leaf_spot       0.63      0.46      0.53       108
Tomato_Tomato_Yellow_Leaf_Curl_Virus       0.72      0.69      0.71        84
                      Tomato_healthy       0.65      0.69      0.67        71
                 Tomato_mosaic_virus       0.00      0.00      0.00         0

                            accuracy                          